In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import pandas as pd
from wsd.load_data import load_data

from linpub.metrics import accuracy

from lineval.disambiguation_eval_backup import ClusterByMeaningModelv3
from lineval.disambiguation_eval_backup import OpenAiWordSenseComparatorv3
from wsd.models import DummyComparator, ClusterByMeaningModel

In [ ]:
X,y = load_data("fr")
for x in X :
    x.context = x.example
size = 400
X, y = X[:size], y[:size]

In [ ]:
openai_api_key = os.getenv('OPEN_AI_API_KEY')
comp = OpenAiWordSenseComparatorv3(openai_api_key,
                                   openai_model='gpt-4o',
                                   thought_process=True)
model1 = ClusterByMeaningModelv3(word_sense_comparator=comp)

In [ ]:
dummy2 = DummyComparator(probability=1)
model2 = ClusterByMeaningModel(comparator=dummy2)

dummy3 = DummyComparator(probability=0)
model3 = ClusterByMeaningModel(comparator=dummy3)

In [ ]:
X, y_pred1 = model1.predict(X, verbose=True)
y_pred2 = model2.predict(X, verbose=True)
y_pred3 = model3.predict(X, verbose=True)

In [ ]:
accuracy(y_pred1, y),accuracy(y_pred2, y), accuracy(y_pred3, y)

In [ ]:
X[2].process

In [ ]:
df = pd.DataFrame(X)
df["pred"] = [c.pred for c in X]
df["process"] = [c.process for c in X]
df['y_pred'] = y_pred1
df['y'] = y
df = df.sort_values(by=["lemma"])
df = df[df["process"] != "lone candidate"]
#reindex
df = df.reset_index(drop=True)
df["same_y"] = df["y"] == df["y"].shift(-1)
df["same_pred"] = df["y_pred"] == df["y_pred"].shift(-1)
df["correct"] = df["same_y"] == df["same_pred"]
df

In [ ]:
from lineval.disambiguation_eval_backup import show_plots
show_plots(df)

In [ ]:
df = df[(df["correct"] == False) | (df["correct"].shift(1) == False)]
df

In [ ]:

df.to_csv("test.csv", index=False)